In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# import dataset
df = pd.read_csv("data.csv")

# drop rows with a missing target -- Keras propagates NaN targets straight into
# the loss, which is what turns loss/mae into nan from epoch 1. Features get
# imputed below, but the target never can be.
n_before = len(df)
df = df.dropna(subset=["yield"]).reset_index(drop=True)
print(f"Dropped {n_before - len(df)} rows with missing yield ({n_before} -> {len(df)})")


In [ ]:
# split features into history, static, and weather
history_cols = [
    "yield_lag1",
    "yield_lag2",
    "yield_lag3",
    "yield_prior_mean",
    "yield_prior_std",
    "n_prior_years",
    "trend_slope",
    "trend_pred",
]
static_cols = [
    "lat",
    "lon",
    "land_sqmi",
    "year",
]
S_cols = history_cols + static_cols
weather_vars = [
    "gdd",
    "precip",
    "tmax_mean",
    "tmax_max",
    "heat_days",
    "et0",
    "radiation",
    "water_balance",
]
weather_cols = [
    f"{var}_t{t}"
    for t in range(14)
    for var in weather_vars
]


In [7]:
# split into train, validation, and test sets - important that we split by year to avoid data leakage
train_df = df[df["year"] <= 2022].copy()
val_df   = df[df["year"].isin([2023, 2024])].copy()
test_df  = df[df["year"] == 2025].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 88935
Validation: 5390
Test: 2695


In [9]:
def make_W(data):
    W = data[weather_cols].to_numpy(dtype=np.float32)
    
    # 112 → 14 × 8
    W = W.reshape(-1, 14, 8)
    
    return W

W_train = make_W(train_df)
W_val   = make_W(val_df)
W_test  = make_W(test_df)

print(W_train.shape)
print(W_val.shape)
print(W_test.shape)

(88935, 14, 8)
(5390, 14, 8)
(2695, 14, 8)


In [10]:
S_train = train_df[S_cols].to_numpy(dtype=np.float32)
S_val   = val_df[S_cols].to_numpy(dtype=np.float32)
S_test  = test_df[S_cols].to_numpy(dtype=np.float32)

print(S_train.shape)

(88935, 12)


In [11]:
y_train = train_df["yield"].to_numpy(dtype=np.float32)
y_val   = val_df["yield"].to_numpy(dtype=np.float32)
y_test  = test_df["yield"].to_numpy(dtype=np.float32)

print(y_train.shape)

(88935,)


In [13]:
weather_imputer = SimpleImputer(strategy="median")

W_train_flat = W_train.reshape(len(W_train), -1)
W_val_flat   = W_val.reshape(len(W_val), -1)
W_test_flat  = W_test.reshape(len(W_test), -1)

W_train_flat = weather_imputer.fit_transform(W_train_flat)
W_val_flat   = weather_imputer.transform(W_val_flat)
W_test_flat  = weather_imputer.transform(W_test_flat)

W_train = W_train_flat.reshape(-1, 14, 8)
W_val   = W_val_flat.reshape(-1, 14, 8)
W_test  = W_test_flat.reshape(-1, 14, 8)

In [15]:
static_imputer = SimpleImputer(strategy="median")

S_train = static_imputer.fit_transform(S_train)
S_val   = static_imputer.transform(S_val)
S_test  = static_imputer.transform(S_test)

In [16]:
# scaling weather features
weather_scaler = StandardScaler()

W_train_2d = W_train.reshape(-1, 8)
W_val_2d   = W_val.reshape(-1, 8)
W_test_2d  = W_test.reshape(-1, 8)

W_train_2d = weather_scaler.fit_transform(W_train_2d)
W_val_2d   = weather_scaler.transform(W_val_2d)
W_test_2d  = weather_scaler.transform(W_test_2d)

W_train = W_train_2d.reshape(-1, 14, 8)
W_val   = W_val_2d.reshape(-1, 14, 8)
W_test  = W_test_2d.reshape(-1, 14, 8)

static_scaler = StandardScaler()

S_train = static_scaler.fit_transform(S_train)
S_val   = static_scaler.transform(S_val)
S_test  = static_scaler.transform(S_test)

yield_scaler = StandardScaler()

y_train_scaled = yield_scaler.fit_transform(
    y_train.reshape(-1, 1)
).ravel()

y_val_scaled = yield_scaler.transform(
    y_val.reshape(-1, 1)
).ravel()

y_test_scaled = yield_scaler.transform(
    y_test.reshape(-1, 1)
).ravel()

In [17]:
weather_input = Input(shape=(14, 8), name="weather")

x = LSTM(64, return_sequences=True)(weather_input)
x = Dropout(0.2)(x)

x = LSTM(32)(x)

In [18]:
static_input = Input(shape=(12,), name="static")

s = Dense(32, activation="relu")(static_input)

In [19]:
combined = Concatenate()([x, s])

combined = Dense(64, activation="relu")(combined)
combined = Dropout(0.2)(combined)

output = Dense(1, name="yield")(combined)

In [20]:
model = Model(
    inputs=[weather_input, static_input],
    outputs=output
)

In [23]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)
#model.summary()

In [24]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [25]:
history = model.fit(
    [W_train, S_train],
    y_train_scaled,

    validation_data=(
        [W_val, S_val],
        y_val_scaled
    ),

    epochs=100,
    batch_size=64,

    callbacks=[early_stopping],

    verbose=1
)

Epoch 1/100
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/100
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/100
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/100
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/100
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 6/100
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 7/100
1125/1390 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: nan - mae: nan

KeyboardInterrupt: 

In [ ]:
pred_test_scaled = model.predict(
    [W_test, S_test]
)
pred_test = yield_scaler.inverse_transform(
    pred_test_scaled
).ravel()

In [53]:
# Self-contained baseline vs model comparison -- rebuilds everything it needs
# from data.csv, so it gives the right answer in a fresh kernel or a stale one.

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error

_df = pd.read_csv("data.csv").dropna(subset=["yield"]).reset_index(drop=True)

_hist   = ["yield_lag1", "yield_lag2", "yield_lag3", "yield_prior_mean",
           "yield_prior_std", "n_prior_years", "trend_slope", "trend_pred"]
_S_cols = _hist + ["lat", "lon", "land_sqmi", "year"]
_wvars  = ["gdd", "precip", "tmax_mean", "tmax_max",
           "heat_days", "et0", "radiation", "water_balance"]
_w_cols = [f"{v}_t{t}" for t in range(14) for v in _wvars]

_train = _df[_df["year"] <= 2022].copy()
_val   = _df[_df["year"].isin([2023, 2024])].copy()
_test  = _df[_df["year"] == 2025].copy()

_mkW = lambda d: d[_w_cols].to_numpy(dtype=np.float32).reshape(-1, 14, 8)
_Wtr, _Wva, _Wte = _mkW(_train), _mkW(_val), _mkW(_test)
_Str = _train[_S_cols].to_numpy(dtype=np.float32)
_Sva = _val[_S_cols].to_numpy(dtype=np.float32)
_Ste = _test[_S_cols].to_numpy(dtype=np.float32)
_ytr = _train["yield"].to_numpy(dtype=np.float32)
_yva = _val["yield"].to_numpy(dtype=np.float32)
_yte = _test["yield"].to_numpy(dtype=np.float32)

_wi  = SimpleImputer(strategy="median")
_Wtr = _wi.fit_transform(_Wtr.reshape(len(_Wtr), -1)).reshape(-1, 14, 8)
_Wva = _wi.transform(_Wva.reshape(len(_Wva), -1)).reshape(-1, 14, 8)
_Wte = _wi.transform(_Wte.reshape(len(_Wte), -1)).reshape(-1, 14, 8)
_si  = SimpleImputer(strategy="median")
_Str, _Sva, _Ste = _si.fit_transform(_Str), _si.transform(_Sva), _si.transform(_Ste)

_ws  = StandardScaler()
_Wtr = _ws.fit_transform(_Wtr.reshape(-1, 8)).reshape(-1, 14, 8)
_Wva = _ws.transform(_Wva.reshape(-1, 8)).reshape(-1, 14, 8)
_Wte = _ws.transform(_Wte.reshape(-1, 8)).reshape(-1, 14, 8)
_ss  = StandardScaler()
_Str, _Sva, _Ste = _ss.fit_transform(_Str), _ss.transform(_Sva), _ss.transform(_Ste)
_ys  = StandardScaler()
_ytr_s = _ys.fit_transform(_ytr.reshape(-1, 1)).ravel()
_yva_s = _ys.transform(_yva.reshape(-1, 1)).ravel()

# reuse the kernel's trained model if it's healthy, otherwise train a fresh one
_m, _pred = globals().get("model"), None
if _m is not None:
    try:
        _pred = _ys.inverse_transform(_m.predict([_Wte, _Ste], verbose=0)).ravel()
        if not np.isfinite(_pred).all():
            _pred = None
    except Exception:
        _pred = None

if _pred is None:
    print("No healthy trained model in this kernel -- training one (~1 min)...")
    _win = Input(shape=(14, 8), name="weather")
    _x = LSTM(64, return_sequences=True)(_win)
    _x = Dropout(0.2)(_x)
    _x = LSTM(32)(_x)
    _sin = Input(shape=(12,), name="static")
    _s = Dense(32, activation="relu")(_sin)
    _c = Concatenate()([_x, _s])
    _c = Dense(64, activation="relu")(_c)
    _c = Dropout(0.2)(_c)
    _m = Model(inputs=[_win, _sin], outputs=Dense(1, name="yield")(_c))
    _m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    _m.fit([_Wtr, _Str], _ytr_s,
           validation_data=([_Wva, _Sva], _yva_s),
           epochs=100, batch_size=64, verbose=0,
           callbacks=[EarlyStopping(monitor="val_loss", patience=10,
                                    restore_best_weights=True)])
    model = _m
    _pred = _ys.inverse_transform(_m.predict([_Wte, _Ste], verbose=0)).ravel()

# persistence baseline: this year's yield = last year's. yield_lag1 is missing
# for ~20% of test rows -- those counties skipped a reporting year, they aren't
# new (n_prior_years is 30+), so walk back down the history chain rather than
# dropping them. Keeps the baseline defined on exactly the rows the model
# predicts on, so the two MAEs are comparable.
_base = (_test["yield_lag1"]
         .fillna(_test["yield_lag2"])
         .fillna(_test["yield_lag3"])
         .fillna(_test["yield_prior_mean"])
         .fillna(np.median(_ytr))          # train-only stat, no test leakage
         .to_numpy())
assert not np.isnan(_base).any()

print(f"Test rows: {len(_yte)}")
print(f"  Baseline MAE: {mean_absolute_error(_yte, _base):.3f}")
print(f"  Model MAE:    {mean_absolute_error(_yte, _pred):.3f}")

# restricted to rows with a real yield_lag1 -- confirms the numbers above
# aren't an artifact of how the missing lags were filled
_mask = _test["yield_lag1"].notna().to_numpy()
print(f"\nRows with a true lag1: {_mask.sum()} / {len(_mask)}")
print(f"  Baseline MAE: {mean_absolute_error(_yte[_mask], _test['yield_lag1'].to_numpy()[_mask]):.3f}")
print(f"  Model MAE:    {mean_absolute_error(_yte[_mask], _pred[_mask]):.3f}")

# expose under the notebook's usual names for any later cells
test_df, y_test, pred_test, baseline_pred = _test, _yte, _pred, _base


Test rows: 1211
  Baseline MAE: 24.121
  Model MAE:    15.311

Rows with a true lag1: 967 / 1211
  Baseline MAE: 23.538
  Model MAE:    14.449


In [ ]:
# ── 2026 forecast map: difference from recent yield ──────────────────────────
# Self-contained: rebuilds everything from data.csv, so it runs in a fresh kernel.
#
# Two deliberate choices behind this map:
#
# 1. AS-OF CUTOFF. 2026 has no labels and a partially observed season: bins
#    t0-t9 are complete, t10 holds only 4 of ~15 days, t11-t13 are empty (see
#    n_days_t*). Running raw 2026 rows through the full-season pipeline would let
#    the median imputer fill Sept/Oct with typical weather from finished years,
#    and the model would read that as real observation. So we truncate to the 10
#    complete bins and TRAIN AT THE SAME CUTOFF. t10 is excluded on purpose: a
#    4-day bin looks like a cold fortnight to a model that only saw 15-day bins.
#
# 2. ANOMALY, NOT ABSOLUTE YIELD. Absolute yield is dominated by the permanent
#    soil-quality gradient, so an absolute map mostly redraws "Iowa has good
#    dirt." Differencing against each county's own recent yield isolates what the
#    SEASON did, which is the part the weather model actually predicts.
#
#    Baseline is the county's mean observed yield over the previous 5 years.
#    NOTE: trend_pred is NOT used as the baseline -- it is biased low by +19 to
#    +37 bu/acre in every year (worst for short-history counties), which would
#    paint the map green regardless of the season. The 5-year county mean is
#    self-centering and needs no bias correction.

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error

sys.path.insert(0, "../pipeline")
import boundaries          # pure-stdlib Census shapefile reader, no geopandas

AS_OF_BIN = 10          # complete biweekly bins -> season through ~mid-August
FORECAST_YEAR = 2026
BASELINE_YEARS = 5
DATA_RAW = Path("../data/raw")

_df = pd.read_csv("data.csv")

_hist   = ["yield_lag1", "yield_lag2", "yield_lag3", "yield_prior_mean",
           "yield_prior_std", "n_prior_years", "trend_slope", "trend_pred"]
_S_cols = _hist + ["lat", "lon", "land_sqmi", "year"]
_wvars  = ["gdd", "precip", "tmax_mean", "tmax_max",
           "heat_days", "et0", "radiation", "water_balance"]
_w_cols = [f"{v}_t{t}" for t in range(AS_OF_BIN) for v in _wvars]

_lab   = _df[_df["yield"].notna()]
_train = _lab[_lab["year"] <= 2023]
_val   = _lab[_lab["year"].isin([2024, 2025])]

# Only ~1,200 of the 2,695 counties in the grid actually report corn, but every
# county gets a 2026 row. Restrict to counties that reported corn in the last
# five years; the rest render as "No Data".
_recent = set(_lab[_lab["year"] >= 2021]["fips"])
_fc = _df[(_df["year"] == FORECAST_YEAR) & _df["fips"].isin(_recent)].copy()

_mkW = lambda d: d[_w_cols].to_numpy(dtype=np.float32).reshape(-1, AS_OF_BIN, 8)
_Wtr, _Wva, _Wfc = _mkW(_train), _mkW(_val), _mkW(_fc)
_Str = _train[_S_cols].to_numpy(dtype=np.float32)
_Sva = _val[_S_cols].to_numpy(dtype=np.float32)
_Sfc = _fc[_S_cols].to_numpy(dtype=np.float32)
_ytr = _train["yield"].to_numpy(dtype=np.float32)
_yva = _val["yield"].to_numpy(dtype=np.float32)

assert not np.isnan(_Wfc).any(), "2026 weather has NaNs -- lower AS_OF_BIN"

_wi  = SimpleImputer(strategy="median")
_Wtr = _wi.fit_transform(_Wtr.reshape(len(_Wtr), -1)).reshape(-1, AS_OF_BIN, 8)
_Wva = _wi.transform(_Wva.reshape(len(_Wva), -1)).reshape(-1, AS_OF_BIN, 8)
_Wfc = _wi.transform(_Wfc.reshape(len(_Wfc), -1)).reshape(-1, AS_OF_BIN, 8)
_si  = SimpleImputer(strategy="median")
_Str, _Sva, _Sfc = _si.fit_transform(_Str), _si.transform(_Sva), _si.transform(_Sfc)

_ws  = StandardScaler()
_Wtr = _ws.fit_transform(_Wtr.reshape(-1, 8)).reshape(-1, AS_OF_BIN, 8)
_Wva = _ws.transform(_Wva.reshape(-1, 8)).reshape(-1, AS_OF_BIN, 8)
_Wfc = _ws.transform(_Wfc.reshape(-1, 8)).reshape(-1, AS_OF_BIN, 8)
_ss  = StandardScaler()
_Str, _Sva, _Sfc = _ss.fit_transform(_Str), _ss.transform(_Sva), _ss.transform(_Sfc)
_ys  = StandardScaler()
_ytr_s = _ys.fit_transform(_ytr.reshape(-1, 1)).ravel()
_yva_s = _ys.transform(_yva.reshape(-1, 1)).ravel()

print(f"Training at {AS_OF_BIN}-bin cutoff ({len(_train):,} rows)...")
tf.random.set_seed(0)
np.random.seed(0)
_win = Input(shape=(AS_OF_BIN, 8), name="weather")
_x = LSTM(64, return_sequences=True)(_win)
_x = Dropout(0.2)(_x)
_x = LSTM(32)(_x)
_sin = Input(shape=(len(_S_cols),), name="static")
_s = Dense(32, activation="relu")(_sin)
_c = Concatenate()([_x, _s])
_c = Dense(64, activation="relu")(_c)
_c = Dropout(0.2)(_c)
model_asof = Model(inputs=[_win, _sin], outputs=Dense(1, name="yield")(_c))
model_asof.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
model_asof.fit([_Wtr, _Str], _ytr_s,
               validation_data=([_Wva, _Sva], _yva_s),
               epochs=100, batch_size=64, verbose=0,
               callbacks=[EarlyStopping(monitor="val_loss", patience=10,
                                        restore_best_weights=True)])

_val_mae = mean_absolute_error(
    _yva, _ys.inverse_transform(model_asof.predict([_Wva, _Sva], verbose=0)).ravel())
_fc["pred"] = _ys.inverse_transform(model_asof.predict([_Wfc, _Sfc], verbose=0)).ravel()
assert np.isfinite(_fc["pred"]).all()

# Anomaly vs the county's own recent history.
_base = (_lab[(_lab["year"] < FORECAST_YEAR)
              & (_lab["year"] >= FORECAST_YEAR - BASELINE_YEARS)]
         .groupby("fips")["yield"].mean())
_fc["baseline"] = _fc["fips"].map(_base)
_fc["anom"] = _fc["pred"] - _fc["baseline"]

# 2026 rows have no NASS record yet, so names are blank -- recover from history.
_names = (_df[_df["state_alpha"].notna()]
          .groupby("fips")
          .agg(state=("state_alpha", "last"), county=("county_name", "last")))
_fc = _fc.join(_names, on="fips")

_ok = _fc.dropna(subset=["anom"])
print(f"Val MAE (2024-25, same cutoff): {_val_mae:.2f} bu/acre")
print(f"Counties mapped: {len(_ok):,}   "
      f"mean predicted yield {_ok['pred'].mean():.1f} bu/acre   "
      f"mean vs recent {_ok['anom'].mean():+.1f}")

# ── map ──────────────────────────────────────────────────────────────────────
# County polygons from Census cartographic boundary files, parsed by
# pipeline/boundaries.py with the standard library only (no geopandas/GDAL --
# see the libomp note in README). Downloads ~1 MB on first run, then caches to
# data/raw/*_boundaries.npz.
_cgeo = boundaries.load("county", DATA_RAW)
_sgeo = boundaries.load("state", DATA_RAW)

_fc["geoid"] = _fc["fips"].astype(int).astype(str).str.zfill(5)
_anom_by_geoid = dict(zip(_fc["geoid"], _fc["anom"]))

_COLORS = ["#d62728", "#ff9e4a", "#8ed17f", "#1a9641"]
_LABELS = [f"below more than 20 bpa", "below less than 20 bpa",
           "above less than 20 bpa", "above more than 20 bpa"]
_cmap = ListedColormap(_COLORS)
_norm = BoundaryNorm([-1e9, -20, 0, 20, 1e9], _cmap.N)

_filled, _fill_colors, _nodata = [], [], []
for _geoid, _rings in _cgeo.items():
    _v = _anom_by_geoid.get(_geoid)
    for _ring in _rings:
        if _v is None or not np.isfinite(_v):
            _nodata.append(_ring)
        else:
            _filled.append(_ring)
            _fill_colors.append(_cmap(_norm(_v)))

fig, ax = plt.subplots(figsize=(15, 9))
ax.add_collection(PolyCollection(_nodata, facecolors="#f2f2f2",
                                 edgecolors="#c8c8c8", linewidths=0.15, zorder=1))
ax.add_collection(PolyCollection(_filled, facecolors=_fill_colors,
                                 edgecolors="#6e6e6e", linewidths=0.2, zorder=2))
ax.add_collection(PolyCollection([r for rs in _sgeo.values() for r in rs],
                                 facecolors="none", edgecolors="#1a1a1a",
                                 linewidths=0.8, zorder=3))

ax.set_xlim(-104, -74)
ax.set_ylim(28, 49.5)
ax.set_aspect(1 / np.cos(np.radians(39)))     # simple equirectangular correction
ax.axis("off")
ax.set_title(f"Difference from Recent Yield in {FORECAST_YEAR}",
             fontsize=15, loc="left", pad=26)
ax.text(0.0, 1.012,
        f"LSTM forecast · season observed through bin {AS_OF_BIN}/14 (~mid-Aug) · "
        f"vs each county's {BASELINE_YEARS}-yr mean · held-out MAE ±{_val_mae:.1f} bpa",
        transform=ax.transAxes, fontsize=9.5, color="#555")

ax.legend(handles=[Patch(facecolor=c, edgecolor="#6e6e6e", label=l)
                   for c, l in zip(_COLORS, _LABELS)]
                  + [Patch(facecolor="#f2f2f2", edgecolor="#c8c8c8", label="No Data")],
          loc="upper left", bbox_to_anchor=(0.0, 0.30), ncol=2,
          fontsize=9, frameon=True, framealpha=0.95)

# Corn Belt summary, acreage-weighted where acres are known.
_belt = _ok[_ok["state"].isin(["IA", "IL", "IN", "OH", "MO",
                               "NE", "KS", "SD", "ND", "MN", "WI", "MI"])]
_w = _belt["acres"].fillna(0).to_numpy()
_belt_pred = (np.average(_belt["pred"], weights=_w) if _w.sum() > 0
              else _belt["pred"].mean())
ax.text(0.995, 0.97,
        f"Corn Belt prediction\n{_belt_pred:.1f} bpa\n"
        f"{_belt['anom'].mean():+.1f} vs recent",
        transform=ax.transAxes, ha="right", va="top", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#ccc"))

plt.tight_layout()
plt.show()

_shares = pd.cut(_ok["anom"], [-np.inf, -20, 0, 20, np.inf],
                 labels=_LABELS).value_counts(normalize=True).reindex(_LABELS)
print("\nCounty share by class:")
for _k, _v in _shares.items():
    print(f"  {_k:<26} {_v:6.1%}")

forecast_2026 = _fc[["fips", "state", "county", "lat", "lon",
                     "pred", "baseline", "anom"]].copy()